# Inspect one MRI from each data split

This notebook loads one deterministic subject from each of the training, validation, and test manifests and displays the axial, coronal, and sagittal center slices of the processed T1 volume. It is a visual data-integrity check, not a model evaluation or a clinical interpretation.

All figure text is in English. Validation and test subjects are shown for inspection only; they must not be used to make training decisions.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd


def find_project_root(start: Path) -> Path:
    """Locate the repository root from the notebook's current directory."""
    for candidate in (start, *start.parents):
        if (candidate / "PROJECT_CONTEXT.md").is_file() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find the project root. Start Jupyter from the repository or a subdirectory."
    )


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SPLIT_DIR = PROJECT_ROOT / "data" / "processed" / "binary_cdr" / "splits"
SPLITS = ("train", "validation", "test")
RANDOM_SEED = 42

if not SPLIT_DIR.is_dir():
    raise FileNotFoundError(
        f"Split manifests not found at {SPLIT_DIR}. "
        "Run `python -m src.data.prepare_binary_splits` from the project root first."
    )

print(f"Project root: {PROJECT_ROOT}")
print(f"Split manifests: {SPLIT_DIR}")

In [ ]:
samples = []

for split in SPLITS:
    manifest_path = SPLIT_DIR / f"{split}.csv"
    if not manifest_path.is_file():
        raise FileNotFoundError(f"Missing {split} manifest: {manifest_path}")

    manifest = pd.read_csv(manifest_path)
    if manifest.empty:
        raise ValueError(f"The {split} manifest contains no subjects.")

    sample = manifest.sample(n=1, random_state=RANDOM_SEED).iloc[0].copy()
    image_path = PROJECT_ROOT / sample["organized_image_path"]
    if not image_path.is_file():
        raise FileNotFoundError(
            f"MRI volume for {sample['participant_id']} is missing or its source disk is not mounted: "
            f"{image_path}"
        )

    sample["split"] = split
    sample["image_path"] = image_path
    samples.append(sample)

pd.DataFrame(
    [
        {
            "Split": sample["split"].title(),
            "Subject": sample["participant_id"],
            "CDR": sample["cdr"],
            "Binary class": "CDR = 0" if int(sample["binary_label"]) == 0 else "CDR > 0",
            "MRI available": sample["image_path"].is_file(),
        }
        for sample in samples
    ]
)

In [ ]:
fig, axes = plt.subplots(nrows=len(samples), ncols=3, figsize=(12, 12))
view_names = ("Axial", "Coronal", "Sagittal")

for row_index, sample in enumerate(samples):
    image = nib.load(str(sample["image_path"]))
    image = nib.as_closest_canonical(image)
    volume = np.asarray(image.dataobj, dtype=np.float32)
    if volume.ndim != 3 or any(size == 0 for size in volume.shape):
        raise ValueError(
            f"Expected a non-empty 3D T1 volume for {sample['participant_id']}; "
            f"got shape {volume.shape}."
        )

    nonzero_finite = volume[np.isfinite(volume) & (volume != 0)]
    if nonzero_finite.size == 0:
        raise ValueError(f"No finite non-zero intensities found in {sample['image_path']}.")
    display_min, display_max = np.percentile(nonzero_finite, (1, 99))
    if display_min >= display_max:
        display_min, display_max = float(nonzero_finite.min()), float(nonzero_finite.max())

    center = tuple(size // 2 for size in volume.shape)
    center_slices = (
        np.rot90(volume[:, :, center[2]]),
        np.rot90(volume[:, center[1], :]),
        np.rot90(volume[center[0], :, :]),
    )
    label = "CDR = 0" if int(sample["binary_label"]) == 0 else "CDR > 0"
    row_title = f"{sample['split'].title()} · {sample['participant_id']} · {label} (CDR {sample['cdr']})"

    for column_index, (view_name, slice_image) in enumerate(zip(view_names, center_slices)):
        axis = axes[row_index, column_index]
        axis.imshow(slice_image, cmap="gray", origin="lower", vmin=display_min, vmax=display_max)
        axis.set_title(f"{row_title}\n{view_name}", fontsize=10)
        axis.axis("off")

    print(
        f"{sample['split'].title()}: {sample['participant_id']} | "
        f"shape={volume.shape} | voxel size (mm)={tuple(round(v, 2) for v in image.header.get_zooms()[:3])}"
    )
    del volume, center_slices, image

fig.suptitle("Processed T1 MRI Center Slices by Data Split", fontsize=16, y=1.01)
fig.tight_layout()
plt.show()